In [1]:
from collections import Counter
import pandas as pd
from sklearn.mixture import GaussianMixture
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage1\dataset1_729×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)
min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)   
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape

(729, 87)

In [3]:
def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage1\benchmark1.xlsx"
reported_structures = extract_numbers(file_path)
print(reported_structures)

for j in range(30, 100, 5):
    j = j/100
    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]
    
    for i in range(3, 7):
        GMM = GaussianMixture(n_components=i, covariance_type='full', init_params = 'k-means++', reg_covar= 1e-5, tol  =  1e-4 , random_state=42) 
        GMM.fit(pca_data)
        labels = GMM.predict(pca_data)
        if len(np.unique(labels)) > 1:
            #silhouette_avg = silhouette_score(new_data, labels)
            #calinski_harabasz_avg = calinski_harabasz_score(new_data, labels)
            print(f"\n k={i}:", f"\nPCA采用解释方差率 k={j}:", f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
        else:
            pass
        data[f'cluster_{i}'] = labels
        dict = {}    
        label_1 = []
        for num  in  reported_structures:
            label = labels[num-1]
            label_1.append(int(label))
        print(label_1)

        for cluster in range(i):
            cluster_samples = data[data[f'cluster_{i}'] == cluster]
            total_in_cluster = len(cluster_samples)
            reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])
            dict[cluster] = (reported_in_cluster, total_in_cluster)
        for value in dict.values():
            if value[0] == 0:
                for key, value in dict.items():
                    formatted_value = f"{value[0]}/{value[1]}"
                    print(f"簇编号{key}: {formatted_value}")

[22, 23, 24, 25, 260, 300, 384, 387, 400, 404, 423, 435, 489, 641, 703]

 k=3: 
PCA采用解释方差率 k=0.3: 
Silhouette score: 0.53 
Calinski-Harabasz score: 836.07
[2, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 2, 2, 2]
簇编号0: 0/54
簇编号1: 6/248
簇编号2: 9/427

 k=4: 
PCA采用解释方差率 k=0.3: 
Silhouette score: 0.51 
Calinski-Harabasz score: 821.60
[2, 2, 2, 1, 3, 1, 1, 1, 1, 2, 3, 1, 2, 2, 2]
簇编号0: 0/54
簇编号1: 6/212
簇编号2: 7/339
簇编号3: 2/124

 k=5: 
PCA采用解释方差率 k=0.3: 
Silhouette score: 0.42 
Calinski-Harabasz score: 899.21
[2, 4, 4, 1, 2, 1, 1, 1, 3, 4, 2, 3, 2, 2, 2]
簇编号0: 0/54
簇编号1: 4/112
簇编号2: 6/196
簇编号3: 2/134
簇编号4: 3/233

 k=6: 
PCA采用解释方差率 k=0.3: 
Silhouette score: 0.42 
Calinski-Harabasz score: 942.00
[2, 4, 4, 1, 5, 1, 1, 1, 3, 4, 5, 3, 2, 4, 2]
簇编号0: 0/54
簇编号1: 4/104
簇编号2: 3/87
簇编号3: 2/141
簇编号4: 4/247
簇编号5: 2/96

 k=3: 
PCA采用解释方差率 k=0.35: 
Silhouette score: 0.40 
Calinski-Harabasz score: 419.60
[1, 1, 1, 2, 1, 2, 2, 2, 2, 1, 1, 2, 1, 1, 1]
簇编号0: 0/53
簇编号1: 9/408
簇编号2: 6/268

 k=4: 
PCA采用解释方差率 k=0.35: 
Silhouett

In [5]:
pca = PCA(n_components= 0.5, random_state=42)
new_data = pca.fit_transform(X)
print(new_data.shape)
ratios = pca.explained_variance_ratio_   
components = pca.components_    
X = pd.DataFrame(X) 
components_df = pd.DataFrame(
    components,
    columns=X.columns, 
    index=[f"PC{i+1}" for i in range(components.shape[0])]  
)
output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)

i = 6
GMM = GaussianMixture(n_components=i, covariance_type='full', init_params = 'k-means++', reg_covar= 1e-5, tol  =  1e-4 , random_state=42)
GMM.fit(new_data)
labels = GMM.predict(new_data)
print(f"Silhouette score: {silhouette_score(new_data, labels):.2f}")
print(f"Calinski-Harabasz score: {calinski_harabasz_score(new_data, labels):.2f}")
df = pd.DataFrame(labels)
df.to_excel("GMM_result.xlsx")
data[f'cluster_{i}'] = labels
        
print(f"\n k={i}:")
for cluster in range(i):
    cluster_samples = data[data[f'cluster_{i}'] == cluster]
    total_in_cluster = len(cluster_samples)       
    reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])
            
    print(f"簇编号 {cluster}: {reported_in_cluster}/{total_in_cluster} "
                f"(已报道比例: {reported_in_cluster/total_in_cluster:.2%})")

(729, 4)
Silhouette score: 0.21
Calinski-Harabasz score: 229.66

 k=6:
簇编号 0: 0/53 (已报道比例: 0.00%)
簇编号 1: 4/154 (已报道比例: 2.60%)
簇编号 2: 0/84 (已报道比例: 0.00%)
簇编号 3: 4/144 (已报道比例: 2.78%)
簇编号 4: 6/187 (已报道比例: 3.21%)
簇编号 5: 1/107 (已报道比例: 0.93%)


In [6]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook

wb = Workbook()
ws = wb.active

for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)
red_bold_font = Font(color="FF0000", bold=True)
for row in range(2, ws.max_row + 1): 

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]
    
    values = [0 if v is None else v for v in values]
    
    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]
    
    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font 

output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"权重系数矩阵已保存到 {output_file}，并标记了绝对值最大的前 10 个数。")

权重系数矩阵已保存到 pca_components_highlighted.xlsx，并标记了绝对值最大的前 10 个数。
